In [68]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# ==============================
# 1️⃣ LOAD DATA
# ==============================
df = pd.read_csv("../Data/coral_Regime.csv")  # Replace with your CSV path

# Replace non-numeric placeholders with NaN
df.replace('nd', np.nan, inplace=True)

# ==============================
# 2️⃣ BIN BLEACHING LEVEL (if needed)
# ==============================
# If you have a numeric column like 'Percent_Bleaching', convert to categories:
# 1️⃣ Convert Percent_Bleaching to numeric
df['Percent_Bleaching'] = pd.to_numeric(df['Percent_Bleaching'], errors='coerce')

# 2️⃣ Now bin into categories
bins = [0, 10, 30, 100]  # thresholds for Low, Medium, High
labels = ['Low', 'Medium', 'High']
df['Bleaching_Level'] = pd.cut(df['Percent_Bleaching'], bins=bins, labels=labels)

# 3️⃣ Drop rows where target is still NaN
df = df.dropna(subset=['Bleaching_Level'])



# ==============================
# 3️⃣ DEFINE FEATURES AND TARGET
# ==============================
feature_cols = [
    'Latitude_Degrees','Longitude_Degrees','Depth_m','Percent_Cover',
    'ClimSST','Temperature_Kelvin','Temperature_Mean','Temperature_Minimum',
    'Temperature_Maximum','Temperature_Kelvin_Standard_Deviation','Windspeed',
    'SSTA','SSTA_Mean','SSTA_Maximum','SSTA_Minimum',
    'TSA','TSA_Mean','TSA_Maximum','TSA_Minimum'
]
target_col = 'Bleaching_Level'

# Convert all feature columns to numeric
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing features
df = df.dropna(subset=feature_cols)

X = df[feature_cols]
y = df[target_col]

# ==============================
# 4️⃣ IMPUTE MISSING FEATURES AND SCALE
# ==============================
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# ==============================
# 5️⃣ TRAIN MODEL
# ==============================
model = RandomForestClassifier(random_state=42)
model.fit(X_scaled, y)

# ==============================
# 6️⃣ SELECT EXAMPLES FOR TESTING
# ==============================
examples = {}
for cls in y.unique():  # pick one row per class
    row = X[y==cls].iloc[0]
    examples[cls] = row

# ==============================
# 7️⃣ PREDICT FUNCTION
# ==============================
def predict_bleaching(sample):
    df_new = pd.DataFrame([sample])
    for col in df_new.columns:
        df_new[col] = pd.to_numeric(df_new[col], errors='coerce')
    X_new = imputer.transform(df_new)
    X_new = scaler.transform(X_new)
    pred_class = model.predict(X_new)[0]
    pred_proba = model.predict_proba(X_new)[0]
    return pred_class, dict(zip(model.classes_, pred_proba))

# ==============================
# 8️⃣ RUN PREDICTIONS
# ==============================
for cls, row in examples.items():
    pred_class, class_prob = predict_bleaching(row)
    print(f"Original Class: {cls}")
    print(f"Predicted Class: {pred_class}")
    print(f"Probability Distribution: {class_prob}")
    print("-"*50)


C:\Windows\Temp\ipykernel_6884\975255166.py:10: DtypeWarning: Columns (13,15,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Data/coral_Regime.csv")  # Replace with your CSV path


Original Class: Low
Predicted Class: Low
Probability Distribution: {'High': np.float64(0.0), 'Low': np.float64(1.0), 'Medium': np.float64(0.0)}
--------------------------------------------------
Original Class: Medium
Predicted Class: Medium
Probability Distribution: {'High': np.float64(0.0), 'Low': np.float64(0.2), 'Medium': np.float64(0.8)}
--------------------------------------------------
Original Class: High
Predicted Class: High
Probability Distribution: {'High': np.float64(0.77), 'Low': np.float64(0.23), 'Medium': np.float64(0.0)}
--------------------------------------------------
